# LMP-SPARK
**Author:** Ryan J. McLaughlin  
**Date:** 2025-05-06

This is meant to be a complete end-to-end document for running:

1. Amplicon Sequence Variant (ASV) pipeline
2. General statistics
3. Downstream analytics
4. Figure/Table creation

## 1. Amplicon Sequence Variant (ASV) pipeline
This section reviews the steps involved in creating ASVs from raw FASTQ data.

### Setup Environments for running the pipeline

In [1]:
%%bash
# Define the environment name
ENV_NAME="spark_env"
ENV_YAML="$PWD/spark_env.yaml"

QI_NAME="qiime2-amplicon-2024.10"

# Check if the environment exists
if mamba env list | grep -q "^${ENV_NAME} "; then
    echo "Environment ${ENV_NAME} already exists."
else
    echo "Environment ${ENV_NAME} does not exist. Creating it..."
    mamba env create -y -n ${ENV_NAME} -f ${ENV_YAML}
fi

# Check if the QIIME2 environment exists
if mamba env list | grep -q "^${QI_NAME} "; then
    echo "Environment ${QI_NAME} already exists."
else
    echo "Environment ${QI_NAME} does not exist. Creating it..."
    mamba env create -y -n ${QI_NAME} -c bioconda qiime2-amplicon-2024.10
fi


# >>>>>>>>>>>>>>>>>>>>>> ERROR REPORT <<<<<<<<<<<<<<<<<<<<<<

    Traceback (most recent call last):
      File "/home/ryan/mambaforge-pypy3/lib/pypy3.9/site-packages/conda/exception_handler.py", line 18, in __call__
        return func(*args, **kwargs)
      File "/home/ryan/mambaforge-pypy3/lib/pypy3.9/site-packages/conda_env/cli/main.py", line 44, in do_call
        exit_code = getattr(module, func_name)(arguments, parser)
      File "/home/ryan/mambaforge-pypy3/lib/pypy3.9/site-packages/conda/cli/main_info.py", line 441, in execute
        print_envs_list(info_dict["envs"], not context.json)
      File "/home/ryan/mambaforge-pypy3/lib/pypy3.9/site-packages/conda/cli/common.py", line 237, in print_envs_list
        disp_env(prefix)
      File "/home/ryan/mambaforge-pypy3/lib/pypy3.9/site-packages/conda/cli/common.py", line 234, in disp_env
        print(fmt % (name, active, prefix))
    BrokenPipeError: [Errno 32] Broken pipe

`$ /home/ryan/mambaforge-pypy3/condabin/mamba list`

  

Environment spark_env already exists.



# >>>>>>>>>>>>>>>>>>>>>> ERROR REPORT <<<<<<<<<<<<<<<<<<<<<<

    Traceback (most recent call last):
      File "/home/ryan/mambaforge-pypy3/lib/pypy3.9/site-packages/conda/exception_handler.py", line 18, in __call__
        return func(*args, **kwargs)
      File "/home/ryan/mambaforge-pypy3/lib/pypy3.9/site-packages/conda_env/cli/main.py", line 44, in do_call
        exit_code = getattr(module, func_name)(arguments, parser)
      File "/home/ryan/mambaforge-pypy3/lib/pypy3.9/site-packages/conda/cli/main_info.py", line 441, in execute
        print_envs_list(info_dict["envs"], not context.json)
      File "/home/ryan/mambaforge-pypy3/lib/pypy3.9/site-packages/conda/cli/common.py", line 237, in print_envs_list
        disp_env(prefix)
      File "/home/ryan/mambaforge-pypy3/lib/pypy3.9/site-packages/conda/cli/common.py", line 234, in disp_env
        print(fmt % (name, active, prefix))
    BrokenPipeError: [Errno 32] Broken pipe

`$ /home/ryan/mambaforge-pypy3/condabin/mamba list`

  

Environment qiime2-amplicon-2024.10 already exists.


### Run the ASV pipeline

In [ ]:
%%bash
MDIR=$(dirname $(which mamba))
source ${MDIR}/../etc/profile.d/conda.sh
conda activate spark_env

./run_vsearch.sh --skip-fastp --skip-merge --skip-filter --skip-concat --skip-derep --skip-denoise --skip-chimera --skip-swarm --skip-nontarget

### Run General Statistics

In [ ]:
%%bash
MDIR=$(dirname $(which mamba))
source ${MDIR}/../etc/profile.d/conda.sh
conda activate spark_env
THREADS="$(nproc --all)"

seqkit stat -a -T -o vsearch_output/stats/fastq_stats.tsv -j ${THREADS} fastq_input/*
seqkit stat -a -T -o vsearch_output/stats/fastp_fastqs.tsv -j ${THREADS} vsearch_output/fastp/*.fastq.gz
seqkit stat -a -T -o vsearch_output/stats/merged_fastqs.tsv -j ${THREADS} vsearch_output/merged/*.fastq
seqkit stat -a -T -o vsearch_output/stats/filtered_fastqs.tsv -j ${THREADS} vsearch_output/filtered/*.fasta
seqkit stat -a -T -o vsearch_output/stats/concat_fastas.tsv -j ${THREADS} vsearch_output/concat/concat.fasta

### Run QIIME2 Taxonomic Classifier

In [ ]:
%%bash
MDIR=$(dirname $(which mamba))
source ${MDIR}/../etc/profile.d/conda.sh
conda activate qiime2-amplicon-2024.10

python qiime_vs_classifier.py \
  --input-fasta SPARK_data/vsearch_output/ASVs/ASV_filtered.micro.fasta \
  --ref-taxonomy SPARK_data/ref_db/silva-138_2-ssu-nr99-tax.qza \
  --ref-seqs SPARK_data/ref_db/silva-138_2-ssu-nr99-seqs-DNA.qza \
  --output-tsv vsearch_output/taxonomy/ASV_SILVA_tax.full-length.vsearch.tsv \
  --stats-output vsearch_output/taxonomy/ASV_SILVA_stats.full-length.vsearch.tsv

### Build Sankey Diagram

In [ ]:
%%bash
MDIR=$(dirname $(which mamba))
source ${MDIR}/../etc/profile.d/conda.sh
conda activate spark_env

python sankey_builder.py

### Plot Metadata

In [47]:
%%bash
MDIR=$(dirname $(which mamba))
source ${MDIR}/../etc/profile.d/conda.sh
conda activate spark_env

python plot_metadata.py

/home/ryan/mambaforge-pypy3/envs/spark_env/lib/python3.11/site-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/home/ryan/mambaforge-pypy3/envs/spark_env/lib/python3.11/site-packages/umap/umap_.py:1865: UserWarning: using precomputed metric; inverse_transform will be unavailable
  warn("using precomputed metric; inverse_transform will be unavailable")
/home/ryan/mambaforge-pypy3/envs/spark_env/lib/python3.11/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


Performed UMAP (precomputed=True, metric='precomputed'). Embedding shape: (129, 2)


/home/ryan/mambaforge-pypy3/envs/spark_env/lib/python3.11/site-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/home/ryan/mambaforge-pypy3/envs/spark_env/lib/python3.11/site-packages/umap/umap_.py:1865: UserWarning: using precomputed metric; inverse_transform will be unavailable
  warn("using precomputed metric; inverse_transform will be unavailable")
/home/ryan/mambaforge-pypy3/envs/spark_env/lib/python3.11/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


Performed UMAP (precomputed=True, metric='precomputed'). Embedding shape: (129, 2)


/home/ryan/Projects/UBC/LMP/SPARK/plot_metadata.py:223: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  sub_df['Type_Group'] = pd.Categorical(sub_df['Type_Group'], [t for t in type_order if t in list(sub_df['Type_Group'])])
/home/ryan/Projects/UBC/LMP/SPARK/plot_metadata.py:309: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  ms_grp_df = metastat_df.groupby(['Type_Group', 'pass_filter'])['sample'].size().reset_index()
/home/ryan/Projects/UBC/LMP/SPARK/plot_metadata.py:311: FutureWarning: The default value of observed=False is deprecated and will change to observ

(465, 129)


### Plot Upset

In [48]:
%%bash
MDIR=$(dirname $(which mamba))
source ${MDIR}/../etc/profile.d/conda.sh
conda activate spark_env

python plot_upset.py
python venn_bubbles.py

/home/ryan/mambaforge-pypy3/envs/spark_env/lib/python3.11/site-packages/upsetplot/data.py:385: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df.fillna(False, inplace=True)
/home/ryan/mambaforge-pypy3/envs/spark_env/lib/python3.11/site-packages/upsetplot/plotting.py:795: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.

### Run Alpha and Beta Diversity

In [ ]:
%%bash
MDIR=$(dirname $(which mamba))
source ${MDIR}/../etc/profile.d/conda.sh
conda activate spark_env

python calc_div.py
python plot_diversity.py

       group1      group2          pval     tstat      pval_adj  significant
0  Oral Rinse  Lung Brush  3.516600e-13  8.753543  1.054980e-12         True
1  Oral Rinse         BAL  1.780988e-02  2.424428  1.780988e-02         True
2  Lung Brush         BAL  7.750208e-10 -6.953842  1.162531e-09         True
       group1  group2      pval     tstat  pval_adj  significant
0  Non-Cancer  Cancer  0.787383 -0.271195  0.787383        False
p-value annotation legend:
      ns: 5.00e-02 < p <= 1.00e+00
       *: 1.00e-02 < p <= 5.00e-02
      **: 1.00e-03 < p <= 1.00e-02
     ***: 1.00e-04 < p <= 1.00e-03
    ****: p <= 1.00e-04

Oral Rinse vs. BAL: t-test independent samples, P_val:2.157e-02 t=2.348e+00
BAL vs. Lung Brush: t-test independent samples, P_val:2.307e-09 t=6.602e+00
Oral Rinse vs. Lung Brush: t-test independent samples, P_val:1.112e-10 t=7.378e+00


/home/ryan/mambaforge-pypy3/envs/spark_env/lib/python3.11/site-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/home/ryan/mambaforge-pypy3/envs/spark_env/lib/python3.11/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


Performed UMAP dimensionality reduction. Embedding shape: (185, 2)


/home/ryan/mambaforge-pypy3/envs/spark_env/lib/python3.11/site-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/home/ryan/mambaforge-pypy3/envs/spark_env/lib/python3.11/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


Performed UMAP dimensionality reduction. Embedding shape: (185, 2)
       group1      group2      pval     tstat  pval_adj  significant
0  Oral Rinse  Lung Brush  0.000489  3.672106  0.001467         True
1  Oral Rinse         BAL  0.111732  1.613217  0.111732        False
2  Lung Brush         BAL  0.018655 -2.393841  0.027983         True
       group1  group2      pval     tstat  pval_adj  significant
0  Non-Cancer  Cancer  0.522788 -0.644495  0.522788        False
p-value annotation legend:
      ns: 5.00e-02 < p <= 1.00e+00
       *: 1.00e-02 < p <= 5.00e-02
      **: 1.00e-03 < p <= 1.00e-02
     ***: 1.00e-04 < p <= 1.00e-03
    ****: p <= 1.00e-04

Oral Rinse vs. BAL: t-test independent samples, P_val:1.050e-01 t=1.641e+00
BAL vs. Lung Brush: t-test independent samples, P_val:1.966e-02 t=2.373e+00
Oral Rinse vs. Lung Brush: t-test independent samples, P_val:4.078e-04 t=3.684e+00


### Run indicspecies (R)

In [79]:
%%bash
MDIR=$(dirname $(which mamba))
source ${MDIR}/../etc/profile.d/conda.sh
conda activate spark_env
Rscript run_indicspecies.R

Loading required package: permute
── Attaching core tidyverse packages ───────────────────��──── tidyverse 2.0.0 ──
✔ dplyr     1.1.4     ✔ readr     2.1.5
✔ forcats   1.0.0     ✔ stringr   1.5.1
✔ ggplot2   3.5.1     ✔ tibble    3.2.1
✔ lubridate 1.9.4     ✔ tidyr     1.3.1
✔ purrr     1.0.4     
── Conflicts ───────────────────────────��────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors


ASV table dimensions: 465 129 
Metadata dimensions: 129 26 


### Plot indicspecies Results

In [80]:
%%bash
MDIR=$(dirname $(which mamba))
source ${MDIR}/../etc/profile.d/conda.sh
conda activate spark_env

python plot_indicspecies.py

/home/ryan/Projects/UBC/LMP/SPARK/plot_indicspecies.py:31: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['log_p'] = -np.log10(df['p.value'])
/home/ryan/Projects/UBC/LMP/SPARK/plot_indicspecies.py:34: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['significance'] = False  # Default color for non-significant
/home/ryan/Projects/UBC/LMP/SPARK/plot_indicspecies.py:37: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value 

(38, 18)
(14, 18)


/home/ryan/Projects/UBC/LMP/SPARK/plot_indicspecies.py:174: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['type_color'] = [x if y == True else 'lightgrey' for x,y in zip(df['type_color'], df['status_significance'])]
/home/ryan/Projects/UBC/LMP/SPARK/plot_indicspecies.py:175: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['type_color'] = ['lightgrey' if ((y == True) & (x == 'lightgrey')) else x


(23, 18)
(10, 18)


### Plot Clustermaps

In [56]:
%%bash
MDIR=$(dirname $(which mamba))
source ${MDIR}/../etc/profile.d/conda.sh
conda activate spark_env

python plot_clustermaps.py

### Run SPIEC-EASI (R)

In [71]:
%%bash
MDIR=$(dirname $(which mamba))
source ${MDIR}/../etc/profile.d/conda.sh
conda activate spark_env

Rscript run_speceasi.R


Attaching package: ‘dplyr’

The following objects are masked from ‘package:stats’:

    filter, lag

The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union


Attaching package: ‘igraph’

The following object is masked from ‘package:SpiecEasi’:

    make_graph

The following object is masked from ‘package:tidyr’:

    crossing

The following objects are masked from ‘package:dplyr’:

    as_data_frame, groups, union

The following object is masked from ‘package:tibble’:

    as_data_frame

The following object is masked from ‘package:rlang’:

    is_named

The following objects are masked from ‘package:stats’:

    decompose, spectrum

The following object is masked from ‘package:base’:

    union

── Attaching core tidyverse packages ───────────────────��──── tidyverse 2.0.0 ──
✔ forcats   1.0.0     ✔ purrr     1.0.4
✔ lubridate 1.9.4     ✔ stringr   1.5.1
── Conflicts ───────────────────────────��────────────── tidyverse_conflicts() ──
✖ lubrida

[1] "Loaded filtered count data from /home/ryan/Projects/UBC/LMP/SPARK_data/vsearch_output/spieceasi/count_data_filtered.RDS"
[1] "Number of rows: 129"
[1] "Number of columns: 465"
[1] "Loaded SpiecEasi object from /home/ryan/Projects/UBC/LMP/SPARK_data/vsearch_output/spieceasi/se_gl_cnt.RDS"
   Min. 1st Qu.  Median    Mean 3rd Qu.    Max. 
 0.1001  0.1123  0.1333  0.1470  0.1664  0.3943 
[1] "igraph object saved to /home/ryan/Projects/UBC/LMP/SPARK_data/vsearch_output/spieceasi/ig_gl.RDS"
[1] "Layout coordinates saved to /home/ryan/Projects/UBC/LMP/SPARK_data/vsearch_output/spieceasi/am_coord.RDS"


Warning messages:
1: The `adjmatrix` argument of `graph_from_adjacency_matrix()` must be symmetric
with mode = "undirected" as of igraph 1.6.0.
ℹ Use mode = "max" to achieve the original behavior. 
2: In layout_nicely(ig_signed) :
  Non-positive edge weight found, ignoring all weights during graph layout.


[1] "Edge list saved to /home/ryan/Projects/UBC/LMP/SPARK_data/vsearch_output/spieceasi/edge_list.csv"
[1] "igraph vertices do not have names. Assigning matching ASV IDs."
[1] "Edge list with ASV IDs saved to /home/ryan/Projects/UBC/LMP/SPARK_data/vsearch_output/spieceasi/edge_list_with_asv_ids.csv"


### Graph Network

In [130]:
%%bash
MDIR=$(dirname $(which mamba))
source ${MDIR}/../etc/profile.d/conda.sh
conda activate spark_env

python graph_network.py

Ignoring fixed x limits to fulfill fixed data aspect with adjustable data limits.
Ignoring fixed y limits to fulfill fixed data aspect with adjustable data limits.
Ignoring fixed x limits to fulfill fixed data aspect with adjustable data limits.
Ignoring fixed y limits to fulfill fixed data aspect with adjustable data limits.
/home/ryan/Projects/UBC/LMP/SPARK/graph_network.py:546: UserWarning: 

The connectionstyle keyword argument is not applicable when drawing edges
with LineCollection.

To make this warning go away, either specify `arrows=True` to
force FancyArrowPatches or use the default values.
Note that using FancyArrowPatches may be slow for large graphs.

  nx.draw_networkx_edges(G, pos,
Ignoring fixed x limits to fulfill fixed data aspect with adjustable data limits.
Ignoring fixed y limits to fulfill fixed data aspect with adjustable data limits.
/home/ryan/Projects/UBC/LMP/SPARK/graph_network.py:609: UserWarning: 

The connectionstyle keyword argument is not applicable when

CalledProcessError: Command 'b'MDIR=$(dirname $(which mamba))\nsource ${MDIR}/../etc/profile.d/conda.sh\nconda activate spark_env\n\npython graph_network.py\n'' returned non-zero exit status 1.